# load packages

In [ ]:
import pandas as pd

# read in input files

## known pathways

In [ ]:
known_path = pd.read_csv('AD_KMI/pathway_score/pathway_annotation/go_ad_pathways.csv')
print(len(known_path.index))
known_path.head()

## omics LR output

In [ ]:
avg_pathway_weights = pd.read_csv('AD_KMI/ML/statistical_models/omics_weights/MULTIOMICS.PATHWAY_SCORES.LR_METRICS.avg_pathway.standard_scaled.go.keep_quest_comb.csv').drop(columns = 'Unnamed: 0')
print(avg_pathway_weights.shape)
avg_pathway_weights.head()

## pathway map

In [ ]:
pathway_map_go = pd.read_csv('AD_KMI/pathway_score/pathway_annotation/go/AD_KMI.ADSP.ROSMAP.all_omics.MSBB.all_omics.go.gene_to_pathway.pathway_mapping.txt',
                             sep = '\t')
print(len(pathway_map_go.index))
print(len(pathway_map_go['PATHWAY_ID'].unique()))
pathway_map_go.head()

## go pathway names

In [ ]:
pathway_ids = []
pathway_names = []
sources = []

with open('AD_KMI/pathway_score/pathway_annotation/raw_databases/go-basic.obo.1', "r") as f:
    for line in f:
        if line.startswith("id: "):
            pathway_ids.append(line)
        elif line.startswith('name: '):
            pathway_names.append(line)
        elif line.startswith('namespace: '):
            sources.append(line)

#print(len(pathway_ids))
#print(len(sources))

go_source = pd.DataFrame({'PATHWAY_ID' : pathway_ids, 'PATHWAY_NAME' : pathway_names, 'SOURCE' : sources})
go_source['PATHWAY_ID'] = go_source['PATHWAY_ID'].str.replace('\n', '')
go_source['PATHWAY_ID'] = go_source['PATHWAY_ID'].str.replace('id: ', '')
go_source['PATHWAY_NAME'] = go_source['PATHWAY_NAME'].str.replace('\n', '')
go_source['PATHWAY_NAME'] = go_source['PATHWAY_NAME'].str.replace('name: ', '')
go_source['SOURCE'] = go_source['SOURCE'].str.replace('\n', '')
go_source['SOURCE'] = go_source['SOURCE'].str.replace('namespace: ', '')
print(len(go_source.index))
print(len(go_source['PATHWAY_ID'].unique()))
print(len(go_source['PATHWAY_NAME'].unique()))
print(go_source['SOURCE'].unique())
go_source.head()

## gene map

In [ ]:
gene_map = pd.read_csv('AD_KMI/pathway_score/pathway_annotation/go/AD_KMI.ADSP.omics_only.GRCh38.113.refseq.exp_validated.go.gene_to_pathway.no_duplicates.pathway_intersection.pathway_mapping.txt',
                       sep = '\t')

## known gene

In [ ]:
known_gene = pd.read_csv('AD_KMI/advp/AD_known_gene_list.txt', header = None)

# create column map

In [ ]:
pathway_map_go_sub = go_source[['PATHWAY_ID', 'PATHWAY_NAME']].drop_duplicates()
print(len(pathway_map_go_sub.index))
print(len(go_source.index))
pathway_map_go_sub.head()

In [ ]:
pathway_map_go_known = pathway_map_go_sub[pathway_map_go_sub['PATHWAY_ID'].isin(known_path['GO_ID'])]
print(len(pathway_map_go_sub['PATHWAY_ID'].unique()))
print(len(pathway_map_go_known['PATHWAY_ID'].unique()))
print(len(known_path['GO_ID'].unique()))

In [ ]:
pathway_map_go_known['AD_BIOLOGICAL_DOMAIN'] = 'Yes'
pathway_map_go_known = pathway_map_go_known.drop(columns = ['PATHWAY_NAME'])

In [ ]:
pathway_map_go_fixed = pathway_map_go_sub.merge(pathway_map_go_known, on = 'PATHWAY_ID', how = 'left')
pathway_map_go_fixed = pathway_map_go_fixed.fillna('No')
print(len(pathway_map_go_fixed.index))
print(len(pathway_map_go_sub.index))
pathway_map_go_fixed.head()

# add pathway info to omics LR output

In [ ]:
omics_lr_output = avg_pathway_weights.rename(columns = {'Feature' : 'PATHWAY_ID'})

In [ ]:
omics_pathway_info = pathway_map_go_fixed.merge(omics_lr_output, on = 'PATHWAY_ID')
print(len(omics_lr_output.index))
print(len(omics_pathway_info.index))
omics_pathway_info.head()

In [ ]:
omics_pathway_info = omics_pathway_info.sort_values(by = 'PVAL')
omics_pathway_info.head()

# add known gene info

In [ ]:
gene_map_known = gene_map[gene_map['GENE'].isin(known_gene[0])][['PATHWAY_ID']].drop_duplicates()
gene_map_known['KNOWN_AD_GENE'] = 'Yes'
print(gene_map_known.shape)
gene_map_known.head()

In [ ]:
omics_known_gene = omics_pathway_info.merge(gene_map_known, on = 'PATHWAY_ID', how = 'left')
omics_known_gene = omics_known_gene.fillna('No')
print(len(omics_pathway_info.index))
print(len(omics_known_gene.index))
omics_known_gene.head()

# add genes

In [ ]:
gene_map = gene_map.sort_values(by = ['PATHWAY_ID', 'GENE'])
gene_map_agg = (gene_map.groupby("PATHWAY_ID")["GENE"].apply(lambda x: ";".join(x.unique())).reset_index())
gene_map_agg = gene_map_agg.rename(columns = {'GENE' : 'GENES'})
print(len(gene_map_agg.index))
print(len(gene_map_agg['PATHWAY_ID'].unique()))
gene_map_agg.head()

In [ ]:
omics_gene = omics_known_gene.merge(gene_map_agg, on = 'PATHWAY_ID', how = 'inner')
print(len(omics_known_gene.index))
print(len(omics_gene.index))
omics_gene.head()

In [ ]:
omics_gene = omics_gene[['PATHWAY_ID', 'PATHWAY_NAME', 'AD_BIOLOGICAL_DOMAIN', 'KNOWN_AD_GENE', 'GENES', 'BETA', 'PVAL']]
omics_gene

# export

In [ ]:
omics_gene.to_csv('AD_KMI/ML/statistical_models/omics_weights/MULTIOMICS.PATHWAY_SCORES.LR_METRICS.for_paper.csv',
                  index = None)